# LitWBTransferLearningYolo notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TXSTCODEPLAYGROUND/CodeCS4337Fall2026/blob/main/LitWBTransferLearningYolo/lit_wb_transfer_learning_yolo_notebook.ipynb)

This notebook trains **LitWBTransferLearningYolo**: YOLOv8n pretrained on COCO, transferred to the IVCNZ pothole dataset (1,243 road photos, one class). What the project does and what it teaches is in its [README](README.md). Every project has its own notebook like this one, in its folder.

The project has **three configs**, one per transfer-learning strategy. This notebook runs all three and compares them:

| Config | Strategy | What is trained |
| --- | --- | --- |
| `config01.json` | Frozen backbone | Neck and the new pothole head. Layers 0 to 9 stay at the COCO weights |
| `config02.json` | Fine-tuning | Every layer, with a smaller learning rate |
| `config03.json` | From scratch | Every layer, from random weights: the baseline |

### Choose your route

| Route | Sections to run | Then run the project with |
| --- | --- | --- |
| **Colab, from notebook cells** | 1 (setup), then 2 and the sections after it | Option A1 (`!python -m ...`) or Option B (`main(...)`) |
| **Colab, from Colab's terminal** (Colab Pro) | 1 (setup) in the notebook first | Option A2: the same commands in the terminal |
| **Locally, from this notebook** | Skip 1, start at [**2. Run the project**](#2.-Run-the-project-(choose-one-option)) | Option A1 or B, with the `.venv` as the notebook's kernel |
| **Locally, from your terminal** | None, no notebook needed | `python -m LitWBTransferLearningYolo --config config01.json` from the repo root, with the `.venv` activated |

In Colab, the setup cells are needed even when you train in the terminal: Google Drive can only be mounted, and Colab Secrets only read, from a notebook cell. Locally, set up the repo once first (`.venv`, requirements, and `.env`, see the main README), and open this notebook from the project folder.

In Colab, run the cells from top to bottom.

1. **Use a GPU:** *Runtime → Change runtime type → GPU*. Training works on a CPU, much more slowly.
2. **Mount Google Drive** so your results survive runtime resets.
3. **Set up the repo:** clone (or update) it, apply the Colab settings, and install only the missing requirements.
4. **Load your API keys** (optional) from Colab Secrets.
5. **Run the three configs**, either as terminal commands or from Python.
6. **Look at the data, the frozen layers, and the metrics** in the teaching sections.
7. **Read the results**, then **load a trained model** (section 4) or **keep training** a run (section 5).

Before a long training run, read the [pro tips](#pro-tips-keep-training-running) at the end.


## 1. Setup

> **Running locally?** Skip this section: these cells only work in Colab. Jump to [**2. Run the project**](#2.-Run-the-project-(choose-one-option)).

Run these cells once per session (and again after the runtime resets): mount Drive, set up the repo, change directory, and load API keys. The setup cell is safe to re-run, e.g. to get new projects with `git pull`; run the API-key cell again after it.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
set -e

cd /content
if [ -d CodeCS4337Fall2026/.git ]; then
  git -C CodeCS4337Fall2026 pull --ff-only
else
  git clone https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git
fi
cd CodeCS4337Fall2026

# Colab settings: data on the local disk, results in Google Drive.
cp .envcolab .env

# Install only packages Colab does not already have (versions are not pinned).
grep -vE '^\s*(#|$)' requirements.txt \
  | sed -E 's/[<>=!~;[ ].*//' \
  | while read -r pkg; do
      if pip show "$pkg" > /dev/null 2>&1; then
        echo "skip $pkg (already installed)"
      else
        pip install -q "$pkg" && echo "installed $pkg"
      fi
    done


In [ ]:
%cd /content/CodeCS4337Fall2026

### Load API keys (optional)

This project logs every run to Weights & Biases (W&B), which needs `WANDB_API_KEY`. Without it, runs are logged to W&B offline and can be uploaded later with the `wandb sync` command printed at the end of the run.

1. Add the keys as Colab **Secrets**: key icon in the left sidebar → *Add new secret*, named `WANDB_API_KEY`. Never type keys into a notebook cell or into `.envcolab`, which is public on GitHub.
2. Run the cell below **after the setup cells and before running the project**, whether you run it from this notebook (Option A1 or B) or from Colab's terminal (A2). Run it again whenever you re-run the setup cell.

The cell loads `.env`, then looks up every key that is still missing in your Secrets (Colab may ask you to *Grant access*). Keys it finds are added to this notebook's environment and saved to `/content/CodeCS4337Fall2026/.env`, so terminal runs find them too. That file is not in git and is deleted when the runtime resets. Values are never printed.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google.colab import userdata

ENV_FILE = Path("/content/CodeCS4337Fall2026/.env")
KEYS = ["WANDB_API_KEY"]

if load_dotenv(ENV_FILE):
    print(f"Loaded settings from {ENV_FILE}")
else:
    print(f"No settings in {ENV_FILE}: run the setup cells first")

for key in KEYS:
    if os.getenv(key):
        print(f"{key}: found in .env")
        continue
    try:
        os.environ[key] = userdata.get(key)
    except Exception:  # noqa: BLE001 (Secret not added or access not granted)
        print(f"{key}: not found, W&B will log offline")
        continue
    with ENV_FILE.open("a") as env_file:
        env_file.write(f"\n{key}={os.environ[key]}\n")
    print(f"{key}: loaded from Colab Secrets and saved to .env")

## 2. Run the project (choose one option)

Both options run exactly the same training and save results to the same folder (Google Drive in Colab, `runs/` locally). Pass a config name from the project's `configs/` folder. The three configs are `config01.json` (frozen backbone), `config02.json` (fine-tuning), and `config03.json` (from scratch). To try your own settings, copy one of them in the file browser (left sidebar, `CodeCS4337Fall2026/LitWBTransferLearningYolo/configs/`), e.g. to `config04.json`, edit it, and pass its name.

**Run the next cell first** (in Colab and locally): it finds the repo folder and makes the project importable.

| Option | How | When to use it |
| --- | --- | --- |
| **A. Terminal command** | `!` in a notebook cell, or Colab's terminal (Colab Pro) | Same command as on your own machine |
| **B. From Python** | `from LitWBTransferLearningYolo import main` | Simple, everything stays in the notebook |


In [ ]:
import sys
from pathlib import Path

# Colab: the clone made by the setup cell. Locally: the parent of this notebook's project folder.
IN_COLAB = "google.colab" in sys.modules
REPO = Path("/content/CodeCS4337Fall2026") if IN_COLAB else Path.cwd().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(f"Repo folder: {REPO}")

### Where your results go

Run the cell below and look at what it prints: `OUTPUT_DIR` is the folder where **every run of this project is saved**: weights, metrics, and `runs_summary.csv`. It is set in `.env` at the repo root.

- **In Colab** it is in your Google Drive, `/content/drive/MyDrive/CodeCS4337Fall2026/runs` (from `.envcolab`), so your results survive runtime resets. Open *MyDrive → CodeCS4337Fall2026 → runs* in Google Drive to find them later.
- **Locally** it is `runs`, i.e. the `runs/` folder of the repo (relative paths are relative to the repo root).

Each run gets its own folder, `<OUTPUT_DIR>/LitWBTransferLearningYolo/<config>/<timestamp>/`. `weights/last.pt` in that folder is rewritten every epoch, and `weights/best.pt` whenever validation mAP50-95 improves, so both are on Drive during training.


In [ ]:
import os

from dotenv import load_dotenv

# Colab: the copy of .envcolab made by the setup cell. Locally: your own .env.
load_dotenv(REPO / ".env")
!echo "OUTPUT_DIR=$OUTPUT_DIR"
output_dir = REPO / os.getenv("OUTPUT_DIR", "runs")
print(f"Runs of this project are saved in: {output_dir / 'LitWBTransferLearningYolo'}")

### Option A: Run as a terminal command

**A1. From a notebook cell (everyone):** a line starting with `!` is sent to the shell, and `{REPO}` is replaced by the repo folder found above. Run the cell below. Locally, if it reports a missing package, the shell is not using the `.venv`'s Python: use Option B, or run the same command in your own terminal with the `.venv` activated.


In [ ]:
!cd "{REPO}" && for c in config01 config02 config03; do python -m LitWBTransferLearningYolo --config $c.json; done

**A2. From Colab's terminal (Colab Pro):** open the terminal from the **Terminal** button at the bottom left of the Colab window, then run:

```bash
cd /content/CodeCS4337Fall2026
python -m LitWBTransferLearningYolo --config config01.json
python -m LitWBTransferLearningYolo --config config02.json
python -m LitWBTransferLearningYolo --config config03.json
```

The terminal runs on the same machine as this notebook, so it uses the same files, packages, and Drive mount. **You still need to run the setup cells above first**: Drive can only be mounted from a notebook cell, and Colab Secrets can only be read from a notebook cell (the API-key cell saves them to `.env`, where the terminal finds them). While training runs in the terminal, the notebook stays free for other work.


### Option B: Run from Python

The project can be imported because the cell at the start of this section added the repo folder to Python's import path. This works in Colab and locally. Each call trains one config, evaluates its best weights on the validation and test sets, and returns the test metrics. Run a single config by keeping only its name in the list.


In [ ]:
from LitWBTransferLearningYolo import main

test_metrics = {}
for config in ["config01.json", "config02.json", "config03.json"]:
    test_metrics[config] = main(config)
print(test_metrics)

## The dataset

The IVCNZ release is one folder of `img-N.jpg` files, each with an `img-N.txt` of YOLO labels (`class x_center y_center width height`, all between 0 and 1). `prepare_pothole` downloads it, shuffles with seed 42, and copies about 80% of the images to train and 10% each to val and test. The cell below draws four test images with those boxes. Green rectangles are the labeled potholes.


In [ ]:
import os

import matplotlib.pyplot as plt
from dotenv import load_dotenv
from matplotlib.patches import Rectangle
from PIL import Image

from LitWBTransferLearningYolo.dataloaders import (
    image_paths,
    prepare_pothole,
    read_yolo_labels,
    yolo_to_box,
)

load_dotenv(REPO / ".env")
data_dir = REPO / os.getenv("DATA_DIR", "data")
dataset_dir = prepare_pothole(data_dir, seed=42).parent
paths = image_paths(dataset_dir, "test")[:4]

fig, axes = plt.subplots(2, 2, figsize=(10, 8), layout="constrained")
for ax, path in zip(axes.flat, paths):
    image = Image.open(path).convert("RGB")
    ax.imshow(image)
    width, height = image.size
    label_file = dataset_dir / "labels" / "test" / f"{path.stem}.txt"
    for _, box in read_yolo_labels(label_file):
        x_min, y_min, x_max, y_max = yolo_to_box(box, width, height)
        ax.add_patch(
            Rectangle(
                (x_min, y_min),
                x_max - x_min,
                y_max - y_min,
                fill=False,
                edgecolor="lime",
                linewidth=2,
            )
        )
    ax.set_title(path.name)
    ax.axis("off")
fig.suptitle("Test images with labeled potholes")
plt.show()

## What transfer learning changes

YOLOv8n is a stack of layers. The backbone (layers 0 to 9) turns the image into feature maps. The neck and head (layers 10 on) mix those maps and predict boxes. COCO has 80 classes and no pothole class, so config01 and config02 keep the backbone's weights and replace the class scores with one score, `pothole`.

Config01 freezes layers 0 to 9 (`"freeze": 10`): those weights stay at COCO, and only the neck and the new head are trained. Config02 trains every layer, with a smaller learning rate. Config03 builds the same stack from `yolov8n.yaml` and trains it from random weights. The cell lists the layers and which config01 freezes.


In [ ]:
from ultralytics import YOLO

from LitWBTransferLearningYolo.models.loading import weights_path

model = YOLO(weights_path("yolov8n.pt"))
print(
    f"{sum(p.numel() for p in model.model.parameters()):,} parameters in the COCO model"
)
print("config01 freezes layers 0-9 (the backbone). config02 trains every layer.")
for index, layer in enumerate(model.model.model):
    role = "frozen in config01" if index < 10 else "trained in config01"
    print(f"{index:2d}  {type(layer).__name__:<16} {role}")

## Metrics

**IoU** (intersection over union) is the overlap of two boxes divided by the area they cover together. mAP50 counts a prediction as correct when IoU is at least 0.50. mAP75 requires 0.75, so the box has to fit more tightly. mAP50-95 averages the score at 0.50, 0.55, ..., 0.95. The cell computes IoU for a perfect overlap, a shift that still passes mAP50, and a miss.


In [ ]:
def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    overlap_w = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    overlap_h = max(0.0, min(ay2, by2) - max(ay1, by1))
    intersection = overlap_w * overlap_h
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    return intersection / (area_a + area_b - intersection)


shift = iou((0, 0, 10, 10), (2, 0, 12, 10))
print(f"perfect overlap: {iou((0, 0, 10, 10), (0, 0, 10, 10)):.2f}")
print(f"box shifted by 2 px: {shift:.2f}  (counts for mAP50, not for mAP75)")
print(f"no overlap: {iou((0, 0, 10, 10), (20, 20, 30, 30)):.2f}")

## 3. Results

Each run is saved under `<OUTPUT_DIR>/LitWBTransferLearningYolo/<config>/<timestamp>/`: in Colab that is Google Drive, `MyDrive/CodeCS4337Fall2026/runs/`; locally, the `runs/` folder of the repo. Each config has a `runs_summary.csv` with one row per finished run.

The cell below prints the **newest run of each config**: strategy, how many epochs it ran, and its test scores. `test_mAP50-95` is the number the comparison uses.


In [ ]:
import csv
import os

from dotenv import load_dotenv

load_dotenv(REPO / ".env")
project_runs = REPO / os.getenv("OUTPUT_DIR", "runs") / "LitWBTransferLearningYolo"
columns = [
    "strategy",
    "epochs_run",
    "test_precision",
    "test_recall",
    "test_mAP50",
    "test_mAP50-95",
]
print(f"{'config':<10}" + "".join(f"{column:>18}" for column in columns))
for config in ["config01", "config02", "config03"]:
    summary = project_runs / config / "runs_summary.csv"
    if not summary.exists():
        print(f"{config:<10}  (no run yet)")
        continue
    with summary.open() as summary_file:
        newest = list(csv.DictReader(summary_file))[-1]
    print(f"{config:<10}" + "".join(f"{newest[column]:>18}" for column in columns))

### In W&B

Open the W&B project `LitWBTransferLearningYolo` (the link is printed at the end of each run). The runs are grouped by config.

- **Learning curves:** `epoch` on X, `val/mAP50-95` on Y. The two pretrained runs should pull away from the run from scratch.
- **Comparison table:** `gain_vs_from_scratch` is test mAP50-95 minus the from-scratch run. Positive means the COCO weights helped.
- **Predictions:** `val_predictions` and `test_predictions`. Sort by `missed` or `false_alarms`, and toggle ground truth and predictions on the image.

How to read each of them is explained in the [README](README.md#reading-the-report).


## 4. Load a trained model

Running a config again trains a **new** model, unless you resume an earlier run (section 5). To use a model you already trained, load it with `load_model`. It finds the run under `<OUTPUT_DIR>/LitWBTransferLearningYolo/` and returns an Ultralytics YOLO model. Nothing is trained.

**Before this section:** you need at least one run of the config, trained in section 2 (in Colab: trained with Drive mounted). Then, in Colab, run the setup cells of section 1. In Colab and locally, run the first cell of section 2, which finds the repo folder.

| Call | What it loads |
| --- | --- |
| `load_model("config02")` | The newest run of `config02`, `weights/best.pt` |
| `load_model("config02/2026-10-02_12-29-09")` | One specific run folder |
| `load_model("config02", which="last")` | The last epoch instead of the best one |
| `load_model("config01")` | The newest frozen-backbone run |

The model prints which file it loaded.


In [ ]:
from LitWBTransferLearningYolo import load_model

model = load_model("config02")

Try the loaded model on four test images. The drawn boxes are the model's predictions at confidence 0.25, not the dataset's labels. After a full training run they should sit on the potholes; after a one-epoch test they will be rough.


In [ ]:
import os

import matplotlib.pyplot as plt
from dotenv import load_dotenv

from LitWBTransferLearningYolo.dataloaders import image_paths, prepare_pothole

load_dotenv(REPO / ".env")
data_dir = REPO / os.getenv("DATA_DIR", "data")
dataset_dir = prepare_pothole(data_dir, seed=42).parent
paths = image_paths(dataset_dir, "test")[:4]
results = model.predict([str(path) for path in paths], conf=0.25, verbose=False)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), layout="constrained")
for ax, path, result in zip(axes.flat, paths, results):
    ax.imshow(result.plot()[:, :, ::-1])
    ax.set_title(f"{path.name}: {len(result.boxes)} predicted")
    ax.axis("off")
plt.show()

## 5. Keep training a run

`load_model` does not train further. To continue a run, pass `resume_from` to `main`.

| The run you point at | What happens |
| --- | --- |
| **Interrupted** (a disconnect mid-run; `weights/last.pt` still has the optimizer) | Ultralytics finishes that run in the **same** folder, from the last saved epoch, up to the run's original number of epochs |
| **Finished** (every epoch done, or early stopping) | A **new** run folder. Training starts from `weights/best.pt` and runs the config's `"epochs"` **more** epochs. Warm-up is turned off |

| Call | What it continues |
| --- | --- |
| `main("config02.json", resume_from="config02")` | The newest run of `config02` |
| `main("config02.json", resume_from="config02/2026-10-02_12-29-09")` | That run folder |
| `main("config02.json", resume_from="config02/2026-10-02_12-29-09/weights/last.pt")` | That weights file |

On the command line: `python -m LitWBTransferLearningYolo --config config02.json --resume-from config02`.

In Colab, mount Drive first (section 1). The run folders are on Drive, which is where `resume_from` looks.


In [ ]:
from LitWBTransferLearningYolo import main

main("config02.json", resume_from="config02")

## Pro tips: keep training running

Colab disconnects runtimes that look idle (no interaction for a while, often around 90 minutes) and also caps how long a runtime can run in total (around 12 hours on the free tier, longer on paid plans). The exact limits vary and are not guaranteed. When the runtime disconnects, training stops and everything under `/content/` (outside Drive) is deleted.

**Save your state in Google Drive**

- Always mount Drive *before* training. Results then go straight to Drive (`OUTPUT_DIR` in `.envcolab`), not to the temporary disk.
- Two weight files are saved to Drive *during* training: the best model so far (`weights/best.pt`), every time validation mAP50-95 improves, and the last epoch (`weights/last.pt`), after every epoch. If the runtime disconnects mid-run, both are kept.
- `runs_summary.csv` and the W&B report are written when the run finishes. To finish an interrupted run, resume it (section 5): it continues in the same folder from `weights/last.pt`.
- Keep your notebook in Drive too: *File → Save a copy in Drive*. Keep copies of configs you edited in Drive, since `/content/CodeCS4337Fall2026` is deleted on reset.

**Avoid idle disconnects**

- Keep the Colab browser tab open, and don't let your computer go to sleep (plug in your laptop and turn off sleep while training).
- Check on the notebook now and then (scroll, look at the progress bars). Running in the terminal does *not* prevent disconnects: the runtime is shared.
- Don't use scripts or browser extensions that fake activity (auto-clickers); they can get your Colab usage restricted.
- With Colab Pro+, *background execution* keeps the runtime alive even after you close the browser.

**Plan your runs**

- Test a new config with few epochs first (e.g. `"epochs": 1`), then start the full run.
- Run the three configs one at a time if your session is short: each `main(...)` call saves its own results, and `config03` (from scratch) takes the longest.
- Prefer several shorter runs over one very long run; each run saves its own results.
- After a disconnect, re-run the setup cells (including the API-key cell), then resume the run with `resume_from` (section 5) instead of starting over.
- When you are done, free the GPU with *Runtime → Disconnect and delete runtime*. It saves your GPU quota (or compute units on paid plans) for next time.
